# 02 会派と発言の中身を調べ、前処理の決まりを作る(下調べ)

**目的**: 会議録の発言を分析に使う前に、取り除くべきもの・直すべきものを見つける。ここで決めた処理は `src/preprocess/` と `scripts/02_preprocess.py` に移した。

**調べたこと**
- 会派 → 政党の対応(会派名の表記ゆれ、中道改革連合などの新しい会派)
- 発言の中の政党名・議員名(党の名前がそのまま入っていると、文の中身ではなく党名で党を見分けてしまうため、隠す処理を作った)
- 委員長の報告・「質疑は終了いたしました」のような手続きの発言(取り除く)
- 議員の名字の一覧(`data/external/member_names.csv` を作った)
- 党ごとに多く使われる言葉(Fightin' Words)

※ 下調べのためのノートブックで、何日かに分けて試しながら書いたため、セルの実行順はそろっておらず、上から順に実行し直すことはできない。前処理の最終版は `scripts/02_preprocess.py` を見てほしい。

In [1]:
import sys, json, re
from pathlib import Path
import pandas as pd
import numpy as np
sys.path.insert(0, "..")
from src.preprocess.filters import filter_speeches, prepare_text, mask_parties
from sklearn.feature_extraction.text import CountVectorizer
from src.analysis.keywords import fightin_words, top_terms
from src.preprocess.members import build_member_table
import src.preprocess.filters as f
from src.preprocess.filters import SURNAME_RE

In [5]:
# 会派 → 政党
GROUP2PARTY = {
    "自由民主党・無所属の会": "自民", "自由民主党": "自民",
    "立憲民主党・無所属": "立憲", "立憲民主・社民・無所属": "立憲", "立憲民主・無所属": "立憲",
    "中道改革連合・無所属": "中道", "公明党": "公明", "日本維新の会": "維新",
    "国民民主党・無所属クラブ": "国民", "国民民主党・新緑風会": "国民",
    "日本共産党": "共産", "れいわ新選組": "れいわ", "参政党": "参政",
    "日本保守党": "保守", "社会民主党": "社民",
    "チームみらい": "みらい", "チームみらい・無所属の会": "みらい",
}

# 読み込み
RAW = Path("../data/raw/kokkai")
dfs = []
for p in sorted(RAW.glob("speech_*.json")):
    with open(p, encoding="utf-8") as f:
        dfs.append(pd.DataFrame(json.load(f)["speechRecord"]))
df = pd.concat(dfs, ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M")

# 除外
q = filter_speeches(df)
q = q[q["date"] < "2026-08-01"].copy()

# 政党(本文を作る前に)
q["party"] = q["speakerGroup"].map(GROUP2PARTY).fillna("その他")

# 本文(マスクあり / なし)
q["text"] = [prepare_text(s, sp, p) for s, sp, p in zip(q["speech"], q["speaker"], q["party"])]
q["text_nomask"] = q["speech"].map(lambda s: prepare_text(s, mask=False))
print("全体:", len(df), "→ 除外後:", len(q))

全体: 194567 → 除外後: 88297


In [3]:
q.groupby(["nameOfHouse", "speakerGroup"]).size().sort_values(ascending=False)

nameOfHouse  speakerGroup
衆議院          立憲民主党・無所属       12640
参議院          立憲民主・社民・無所属      6830
衆議院          国民民主党・無所属クラブ     6559
             自由民主党・無所属の会      5837
参議院          国民民主党・新緑風会       5659
衆議院          中道改革連合・無所属       5275
             日本維新の会           5084
参議院          公明党              4916
             立憲民主・無所属         4742
             日本維新の会           4676
             日本共産党            4303
衆議院          日本共産党            3154
参議院          自由民主党            2396
             参政党              2161
             自由民主党・無所属の会      1981
衆議院          公明党              1976
             参政党              1916
             れいわ新選組           1843
参議院          れいわ新選組           1794
衆議院          チームみらい           1244
参議院          各派に属しない議員         948
衆議院          有志の会              745
参議院          沖縄の風              531
             日本保守党             389
             社会民主党             388
衆議院          無所属               248
参議院          チームみらい・無所属の会      201
衆議院          日本保守党           

In [4]:
print(mask_parties("国民民主党・新緑風会の河合です。N国党と自公政権。"))

政党の河合です。政党と政党政権。


In [3]:
GROUP2PARTY = {
    "自由民主党・無所属の会": "自民", 
    "自由民主党": "自民",
    "立憲民主党・無所属": "立憲", 
    "立憲民主・社民・無所属": "立憲", 
    "立憲民主・無所属": "立憲",
    "中道改革連合・無所属": "中道", 
    "公明党": "公明", 
    "日本維新の会": "維新",
    "国民民主党・無所属クラブ": "国民", 
    "国民民主党・新緑風会": "国民",
    "日本共産党": "共産", 
    "れいわ新選組": "れいわ", 
    "参政党": "参政",
    "日本保守党": "保守", 
    "社会民主党": "社民",
    "チームみらい": "みらい", 
    "チームみらい・無所属の会": "みらい"
}
q["party"] = q["speakerGroup"].map(GROUP2PARTY).fillna("その他")
q["party"].value_counts()

party
立憲     24353
国民     12293
自民     10217
維新      9770
共産      7457
公明      6901
中道      5297
参政      4088
れいわ     3637
その他     2661
みらい     1451
保守       589
社民       388
Name: count, dtype: int64

In [12]:
# 分析対象の政党の発言から2万件を取り出す
s = q[q["party"] != "その他"].sample(20000, random_state=0)

# 発言 × 2〜4文字の並び の出現回数の表
vec = CountVectorizer(analyzer="char", ngram_range=(2, 4), min_df=20)
X = vec.fit_transform(s["text"])
terms = np.array(vec.get_feature_names_out())

# 党ごとの特徴語(Fightin' Words の z 値の上位15個)
for party, words in top_terms(X, terms, s["party"]).items():
    print(party, ":", "/".join(words))

みらい : ａｉ/えており/考えてお/えてお/デー/ござ/ござい/ございま/ざい/ざいま/いった/データ/ータ/ざいます/ういった
れいわ : です/費税/消費税/障害/すよ/障害者/よ。/すか/沖縄/ですよ/すか。/すよ。/です。/んです/ですよ。
中道 : イラン/イラ/大会/こと。/ること。/教会/一教/一教会/統一教会/統一教/情勢/自転/自転車/いをいた/いをい
保守 : 難民/国人/原子力規/力規制/子力規制/子力規/外国/力規/外国人/太陽光/陽光/発電/ｂｔ/太陽/民認
公明 : まして/、また/伺い/ており/おりま/ります/ておりま/りま/おり/おります/まして、/ます。/ります。/てお/ます
共産 : です/すよ/ですよ/ね。/んですよ/んです/すよ。/よ。/すよね/ですよ。/すか。/よね/ですか。/よね。/すよね。
参政 : 外国/外国人/国人/国民/と考え/と考/日本/考え/、国/ではな/、外国/日本人/はなく、/えます。/では
国民 : いうとこ/うところ/うとこ/のか/というと/りが/ありがと/りがとう/りがと/がとう/がとうご/ところ/いうと/とうご/とうござ
社民 : ｕｓ/ｆｉ/堅持/銀行/依存症/存症/ですか。/すか。/同意が/ャンブル/ャンブ/ギャンブ/ギャン/サー/依存
立憲 : です/すか/ですか/すか。/んです/か。/んで/こと。/ですか。/すよ/ない/ること。/大臣/こと。 /ですよ
維新 : なと/かなと/ですけ/んです/んで/改革/すけ/ですけれ/かな/んですけ/副首都/尖閣/副首/れども/けれども
自民 : でありま/であり/あります/ます。/であ/ります/ります。/あり/ます/りま/ありま/まして/ます。 /ござい/ざいま


In [14]:
names = set()
for g in q["speakerGroup"].unique():
    names.add(g)
    names.update(p for p in g.split("・") if len(p) >= 2)   # 「新緑風会」「無所属クラブ」など

left = {n: q["text"].str.contains(n, regex=False).sum() for n in names}
pd.Series(left)[lambda s: s > 0].sort_values(ascending=False)

国民の声         211
有志の会         159
無所属          133
沖縄の風         110
各派に属しない議員     44
改革の会           6
無所属の会          3
減税保守こども        2
dtype: int64

In [16]:
head = q["text"].str[:80]
intro = head.str.extract(r"([^\s、。]+?)の[^\s、。]{2,8}(?:です|でございます)。")[0]
intro.value_counts().head(30)

0
政党                 7348
有志                   70
沖縄                   64
政党会派                 60
中道                   52
そ                    33
最後                   27
次                    24
愛媛県選出                19
衆議院議員                16
政党代表                 16
無所属                  15
政党・政党                11
政党政調会長               10
参議院幹事長               10
続いて                   9
政党・政党・無所属会派           9
石川県能登半島               9
政党・政党・無所属             9
広島県                   8
二点目                   8
こ                     7
私                     6
参議院議員                 6
東京十六区選出               5
茨城県選出                 5
神奈川県選出                5
参議院                   5
長野五区                  5
お手元に配付されております案文       5
Name: count, dtype: int64

In [ ]:
content = np.array([bool(re.search(r"[一-龥ァ-ヶ]", t)) for t in terms])

for party in ["立憲", "国民", "自民", "維新", "共産", "公明", "中道", "参政", "れいわ"]:
    z = fightin_words(X, (s["party"] == party).to_numpy())
    z[~content] = -np.inf                      # ひらがなだけの並びは除く
    top = terms[np.argsort(z)[::-1][:100]]
    print(party, ":", "/".join(top), "\n")

立憲 : 大臣/大臣、/臣、/修正/皆さん/皆さ/、大臣、/通報/言っ/答え/委員/の皆さん/の皆さ/、大臣/修正案/分か/年金/職員/の修正/私は/労働/の修/関税/員会/言って/公益通報/益通/益通報/公益通/委員会/報者/通報者/僕は/努める/夫婦別姓/婦別姓/お答/お答え/捜査/資料/及び/夫婦/総理/価高対策/る附/の修正案/る附帯/高対策/価高対/物価高対/願いしま/努めるこ/言ってい/決議/る附帯決/発言/言った/基金/電磁的/磁的/残念/分から/うに思い/会計/育士/保育士/大臣。/臣。/皆さんに/答えて/正案/。大臣/お答えく/答えく/答えくだ/思うん/。 十/ 十/的記録/磁的記録/電磁的記/磁的記/電磁/に思いま/的記/分からな/に思い/夫婦別/婦別/法務/思うんで/員の/に思うん/答弁/報告/思う/石破/と言っ/労死/過労死 

国民 : 賦課金/賦課/氷河/河期/氷河期/課金/エネ賦/ネ賦課/ネ賦/再エネ賦/エネ賦課/ネ賦課金/河期世代/期世代/期世/氷河期世/河期世/電気/お教え/テリ/お教/ェンス/手取り/インテリ/ンテリ/テリジ/テリジェ/ンテリジ/手取/戸籍/就職氷/職氷/職氷河期/職氷河/就職氷河/尋ねし/お尋ねし/ジェンス/リジェン/リジェ/リジ/インテ/備蓄/再エネ/再エ/所得制/所得制限/御説明/御説/教え/お教えい/指定都市/指定都/定都市/定都/教えいた/教えい/得制限/得制/手取りを/就職/的基盤/山形/質問です/鉄道/統制/問です。/問です/被爆/解をお伺/デミー/カデミー/ェン/安定供/定供/定供給/安定供給/と思って/た点/浄化/お尋/お尋ね/ジェン/サポー/的基/浄化槽/化槽/伺い/問で/サポ/見解/中で/尋ねしま/質問で/尋ね/課金の/の再エ/の再エネ/取りを増/りを増や 

自民 : 地域/し上げま/告申/告申し/告申し上/報告申し/報告申/御報告申/御報告/御報/上げます/法律案/律案/上げま/本案/地元/申し上げ/申し上/し上げ/し上/申し/重要/本案は/我が国/歯科/の地/律案は/法律案は/原案/取組/を聴取/災害/を聴/会にお/が国/皆様方/終局/様方/いと存/いと存じ/を御報/を御報告/、地/たいと存/離島/員会にお/本案は、/、御報告/査の経/、御報/上、御/さん方/ 本案/。 本案/査の経過/

In [19]:
# 党ごとに最大3000件ずつ(小さな党は全件)
s = (q[q["party"] != "その他"]
     .sample(frac=1, random_state=0)      # 全体をランダムに並べ替える
     .groupby("party")
     .head(3000))                         # 党ごとに先頭3000件
print(s["party"].value_counts())

vec = CountVectorizer(analyzer="char", ngram_range=(2, 4), min_df=20)
X = vec.fit_transform(s["text"])
terms = np.array(vec.get_feature_names_out())
content = np.array([bool(re.search(r"[一-龥ァ-ヶ]", t)) for t in terms])

for party in s["party"].value_counts().index:      # 全党
    z = fightin_words(X, (s["party"] == party).to_numpy())
    z[~content] = -np.inf
    top = terms[np.argsort(z)[::-1][:100]]
    print(party, ":", "/".join(top), "\n")

party
公明     3000
立憲     3000
中道     3000
国民     3000
自民     3000
共産     3000
れいわ    3000
維新     3000
参政     3000
みらい    1451
保守      589
社民      388
Name: count, dtype: int64
公明 : たいと思/伺い/いと思い/思ってお/いと思/取組/思います/ふうに思/思いま/思い/うに思/に思/に思って/うに思っ/伺いたい/伺いた/に思っ/思って/大変に/支援/と思いま/と思い/変に/重要で/大事/思っ/取り組/お伺い/お伺/り組/重要/を伺い/しくお願/くお願い/くお願/を伺/伺いを/お伺いを/大変にあ/変にあ/変にあり/要で/組んで/見解を伺/解を伺/り組んで/解を伺い/組ん/り組ん/取り組ん/取組を/アレル/教師/を伺いた/組を/お声/の取組/要であ/伺いま/今後/組んでい/見解を/患者さ/患者さん/レルギ/アレルギ/レルギー/見解/、大変に/願いした/アレ/伺います/要であり/レル/うに思い/に思い/御見解/に思いま/続きまし/お声を/御見/続きま/伺いをし/活用/重要であ/ 続きま/解を/。 続き/ 続き/お願いを/伺いをさ/て伺い/栄養/て伺/いを申し/いを申/願いを/願い/対策/お願 

立憲 : 大臣/修正/大臣、/臣、/通報/年金/皆さん/皆さ/労働/公益通/公益通報/益通報/関税/益通/電磁的記/磁的記録/的記録/磁的記/磁的/電磁的/的記/、大臣、/言っ/電磁/の皆さん/の皆さ/通報者/報者/の修正/答え/修正案/商品券/品券/米券/お米券/の修/組合/公立病院/公立病/残念/組合の/分か/、大臣/の修正案/る附帯/婦別姓/夫婦別姓/る附/夫婦/る附帯決/臣。/大臣。/額療養費/委員/厚生年/生年金/厚生年金/生年/転嫁/額療/高額療/高額療養/額療養/立病/立病院/総理/格転嫁/価格転嫁/価格転/格転/療養/言って/相互関/互関/養費/療養費/受刑/、労働/高額/互関税/相互関税/婦別/夫婦別/、労/議院に/底上/底上げ/受刑者/刑者/お答/お答え/別姓/お米/言ってい/分から/益通報者/人目/事業再生/ふうに思/言った 

中道 : イラン/イラ/答弁/答弁を/弁を/金利/病歴/

In [6]:
rep = q[q["speech"].str.contains("ご報告申し上げます", regex=False)]
print(len(rep))
rep["speech"].str[:30].value_counts().head(15)

0


Series([], Name: count, dtype: int64)

In [10]:
rep = q[q["speech"].str.contains("御報告申し上げます", regex=False)]
print(len(rep))
rep["speech"].str[:100].value_counts().head(15)

328


speech
○阿久津幸彦君　ただいま議題となりました法律案につきまして、財務金融委員会における審査の経過及び結果を御報告申し上げます。\r\n　本案は、現下の揮発油の価格の高騰から国民生活及び国民経済を守るため、揮発      2
○村井委員　令和七年度の衆議院歳出予算の要求について、庶務小委員会における審査の経過及び結果について御報告申し上げます。\r\n　令和七年度の本院予定経費要求額は六百八十一億一千三百万円余でありまして、こ      1
○青柳（陽）委員　令和七年度の国立国会図書館歳出予算の要求について、図書館運営小委員会における審査の経過及び結果について御報告申し上げます。\r\n　令和七年度の国立国会図書館の予定経費要求額は二百四億二      1
○牧島委員　第一分科会について御報告申し上げます。\r\n　その詳細につきましては会議録に譲ることとし、ここでは主な質疑事項について申し上げます。\r\n　まず、内閣所管については、大阪・関西万博に係る安全対    1
○田所委員　第二分科会について御報告申し上げます。\r\n　本分科会は、総務省所管について審査を行いました。\r\n　詳細につきましては会議録に譲ることといたしますが、その主な質疑事項は、地方創生二・〇に向け    1
○山下委員　第三分科会について御報告申し上げます。\r\n　本分科会は、法務省、外務省及び財務省所管について審査を行いました。\r\n　詳細につきましては会議録に譲ることといたしますが、その主な質疑事項は、難    1
○高木委員　第四分科会について御報告申し上げます。\r\n　本分科会は、文部科学省所管について審査を行いました。\r\n　詳細につきましては会議録に譲ることといたしますが、その主な質疑事項は、学校施設の環境改    1
○深澤委員　第五分科会について御報告申し上げます。\r\n　本分科会は、厚生労働省所管について審査を行いました。\r\n　詳細につきましては会議録に譲ることといたしますが、その主な質疑事項は、年金制度全体に対    1
○小林（茂）委員　第六分科会について御報告申し上げます。\r\n　本分科会は、農林水産省及び環境省所管について審査を行いました。\r\n　詳細につきましては会議録に譲ることといたしますが、その主な質疑事項は、    1
○齋藤

In [13]:
REPORT = r"^[^\n]{0,200}?御報告(?:申し上げ|いたし)"
is_rep = q["speech"].str.contains(REPORT)
print("該当:", is_rep.sum())
print(q[is_rep]["party"].value_counts())
q[is_rep]["speech"].str[:60].sample(10, random_state=0)

該当: 329
party
自民    215
立憲     62
公明     36
維新     10
国民      5
中道      1
Name: count, dtype: int64


82997     ○山田太郎君　ただいま議題となりました法律案につきまして、地方創生及びデジタル社会の形成等に...
50024     ○和田政宗君　ただいま議題となりました法律案につきまして、内閣委員会における審査の経過と結果...
89807     ○堂故茂君　ただいま議題となりました法律案につきまして、文教科学委員会における審査の経過と結...
159307    ○生稲晃子君　ＯＤＡ調査派遣第四班について御報告いたします。\r\n　当班は、本年一月五日か...
92447     ○滝沢求君　ただいま議題となりました条約二件につきまして、外交防衛委員会における審査の経過と...
50009     ○三宅伸吾君　ただいま議題となりました法律案につきまして、財政金融委員会における審査の経過及...
173541    ○藤井比早之君　ただいま議題となりました両法律案につきまして、農林水産委員会における審査の経...
14240     ○三宅伸吾君　ただいま議題となりました両法律案につきまして、財政金融委員会における審査の経過...
87738     ○柘植芳文君　ただいま議題となりました法律案につきまして、厚生労働委員会における審査の経過と...
89835     ○舞立昇治君　ただいま議題となりました法律案につきまして、農林水産委員会における審査の経過と...
Name: speech, dtype: str

In [16]:
has = q["text"].str.contains("中道", regex=False)
print("中道を含む発言:", has.sum())
print(q[has]["party"].value_counts())
q[has & (q["party"] == "自民")]["text"].str.extract(r"(.{0,15}中道.{0,15})")[0]

中道を含む発言: 187
party
中道     104
公明      19
国民      16
自民      13
立憲       9
維新       9
共産       6
れいわ      3
みらい      3
その他      2
参政       2
社民       1
Name: count, dtype: int64


472       し、国民世論は左右両極に分化、中道勢力が縮小していくことで、政治
13138     ございます。そういった意味で、中道的な日本のエネルギー政策という
13190     ただきました。その上で、日本は中道的な対応を続けていくことが大切
126170    が始まった矢先の二月二十四日、中道の小川代表は代表質問の冒頭でこ
141180    た。政党と政党さんが合併をして中道になった。そして、政党、みらい
156035    私が調べたところだと、いわゆる中道左派と呼ばれるような方でござい
171341    というふうに思います。 先ほど中道の國重幹事の方から非常に大事な
172935    でございます。 また、一方で、中道、国民さんの案に対しましても、
174628    党の考えを申し上げたところで、中道提出者、参政提出者にお伺いいた
174752    、立場の隔たりという意味では、中道、国民の皆様からは規制強化の法
176538    ろであります。 これに対して、中道、国民提出の法案は、企業・団体
181129    考人から、欧州統合を支えてきた中道右派、中道左派の二大勢力に代わ
183139    する各党協議会は、政党、政党、中道、国民、参政、みらい、政党を始
Name: 0, dtype: str

In [17]:
AMBIG = "中道|国民|参政|みらい"   # 一般の言葉と重なる党の略称

# 後ろの文脈で判断:「中道提出」「国民さん」「中道の小川代表」など
CTX_AFTER = re.compile(rf"({AMBIG})(?=提出|さん|案|の[一-龥]{{1,4}}(?:代表|幹事長|幹事|議員|委員|提出者))")
# 並びで判断:「政党、中道」「中道、政党」
LIST_BEFORE = re.compile(rf"(?<=政党、)({AMBIG})")
LIST_AFTER = re.compile(rf"({AMBIG})(?=、政党)")


def mask_ambiguous(text):
    """一般の言葉と重なる党の略称を、党を指す文脈のときだけマスクする"""
    text = CTX_AFTER.sub("政党", text)
    for _ in range(5):                    # 「政党、中道、国民、参政」のような連続に対応するため、くり返す
        new = LIST_AFTER.sub("政党", LIST_BEFORE.sub("政党", text))
        if new == text:
            break
        text = new
    return text

In [19]:
titles = q["text"].str.findall(r"([一-龥]{1,3})(?:総理|大臣|代表|幹事長|官房長官|さん)")
pd.Series([n for lst in titles for n in lst]).value_counts().head(40)

皆      12703
高市      2320
石破      1960
外務      1230
今       1158
政党      1143
患者      1074
赤澤      1047
財務      1026
法務       847
副        836
総務       758
防衛       694
内閣       676
小泉       670
生労働      606
子        579
厚労       511
片山       493
茂木       470
加藤       439
総理       405
経産       398
松本       376
文科       361
環境       359
林        355
上野       351
是非       343
福岡       338
鈴木       333
黄川田      320
城内       311
平        305
母        302
小野田      295
岩屋       286
部科学      282
武藤       272
三原       265
Name: count, dtype: int64

In [20]:
titles = q["text"].str.findall(r"([一-龥]{1,3})(?:委員|議員|君)")
pd.Series([n for lst in titles for n in lst]).value_counts().head(40)

予算     1934
国会     1383
教育      526
本       506
衆議院     467
家公安     423
当       379
参議院     358
内閣      354
正取引     351
法務      318
生労働     299
第三者     271
特別      263
決算      213
総務      212
規制      212
地方      210
報保護     208
各       203
院予算     190
専門      186
経営      181
力規制     176
林水産     166
外務      151
務金融     148
環境      145
土交通     144
済産業     138
同僚      131
厚労      127
挙管理     125
交防衛     121
市早苗     115
品安全     111
別撤廃     110
評価      107
報監理     106
政金融     106
Name: count, dtype: int64

In [23]:
# 氏名(発言者の列から)
full_names = sorted(df["speaker"].dropna().unique(), key=len, reverse=True)
full_names = [n for n in full_names if n != "会議録情報" and len(n) >= 3]
print("氏名の数:", len(full_names))

# 名字(衆議院の見出し「○長妻委員」「○赤澤国務大臣」から)
HEAD = re.compile(
    r"^○([一-龥々ヶぁ-んァ-ヶー]{1,6})(?:（[^）]*）)?"
    r"(?:委員|君|国務大臣|内閣総理大臣|副大臣|大臣政務官|政府特別補佐人)"
)
sur = df["speech"].str.extract(HEAD)[0]
pairs = pd.DataFrame({"speaker": df["speaker"], "surname": sur}).dropna().drop_duplicates()

# 氏名の先頭と一致する名字だけを残す(見出しの読み取り誤りを除く)
pairs = pairs[[n.startswith(s) for n, s in zip(pairs["speaker"], pairs["surname"])]]
surnames = sorted(pairs["surname"].unique(), key=len, reverse=True)
print("名字の数:", len(surnames))
print(pairs.head(20))

氏名の数: 2177
名字の数: 983
    speaker surname
1       安住淳      安住
4     小野寺五典     小野寺
5       石破茂      石破
7       城内実      城内
9      武藤容治      武藤
23      江藤拓      江藤
29    中曽根康隆     中曽根
34      岩屋毅      岩屋
36      中谷元      中谷
45     木原誠二      木原
52     赤澤亮正      赤澤
63     塩崎彰久      塩崎
72     加藤勝信      加藤
74     福岡資麿      福岡
87      長妻昭      長妻
132     城井崇      城井
139    あべ俊子      あべ
168   源馬謙太郎      源馬
202      階猛       階
241   奥野総一郎      奥野


In [22]:
members = set(df.loc[df["speakerGroup"].notna(), "speaker"])
no_surname = sorted(members - set(pairs["speaker"]))
print("議員の数:", len(members))
print("名字が取れなかった議員:", len(no_surname))
print(no_surname[:40])

議員の数: 895
名字が取れなかった議員: 74
['あかま二郎', 'あべ俊子', 'いんどう周作', 'うるま譲司', 'おおたけりえ', 'おおつき紅葉', 'かごしま彰宏', 'かまやち敏', 'こうらい啓一郎', 'こやり隆史', 'たがや亮', 'とかしきなおみ', 'なかやめぐ', 'ながえ孝子', 'ラサール石井', '三上えり', '上野ほたる', '世耕弘成', '中曽根弘文', '中条きよし', '伊勢崎賢治', '佐々木さやか', '佐々木りえ', '佐々木雅文', '初鹿野裕樹', '原田大二郎', '吉川ゆうみ', '吉良よし子', '嘉田由紀子', '堂込麻紀子', '塩村あやか', '大島九州男', '大椿ゆうこ', '大門実紀史', '奥田ふみよ', '小島とも子', '小林さやか', '小林孝一郎', '山内佳菜子', '山谷えり子']


In [24]:
full_as_sur = pairs[pairs["speaker"] == pairs["surname"]]
print("氏名まるごとが名字になっている:", len(full_as_sur))
full_as_sur.head(10)

氏名まるごとが名字になっている: 516


,speaker,surname
379,竹谷とし子,竹谷とし子
383,浅田均,浅田均
387,川合孝典,川合孝典
393,小池晃,小池晃
396,熊谷裕人,熊谷裕人
399,比嘉奈津美,比嘉奈津美
403,山本太郎,山本太郎
436,重徳和彦,重徳和彦
441,西岡秀子,西岡秀子
446,斉藤鉄夫,斉藤鉄夫


In [25]:
no_surname = sorted(members - set(pairs.loc[pairs["speaker"] != pairs["surname"], "speaker"]))
print("名字が取れなかった議員:", len(no_surname))

known = sorted(set(pairs.loc[pairs["speaker"] != pairs["surname"], "surname"]), key=len, reverse=True)

extra = []
for name in no_surname:
    for s in known:
        if name.startswith(s) and len(s) < len(name):
            extra.append((name, s))
            break

extra = pd.DataFrame(extra, columns=["speaker", "surname"])
print("補えた議員:", len(extra), "/", len(no_surname))
print(extra.head(20))
print("それでも残った議員:", sorted(set(no_surname) - set(extra["speaker"])))

名字が取れなかった議員: 273
補えた議員: 137 / 273
   speaker surname
0      上田勇      上田
1     上田清司      上田
2    上野ほたる      上野
3     上野通子      上野
4     下野六太      下野
5    中曽根弘文     中曽根
6     中田優子      中田
7     中西祐介      中西
8     井上哲士      井上
9     井上義行      井上
10    伊藤孝恵      伊藤
11    伊藤孝江      伊藤
12     伊藤岳      伊藤
13    伊藤辰夫      伊藤
14    伊藤達也      伊藤
15  佐々木さやか     佐々木
16   佐々木りえ     佐々木
17   佐々木雅文     佐々木
18    佐藤信秋      佐藤
19     佐藤啓      佐藤
それでも残った議員: ['いんどう周作', 'かごしま彰宏', 'かまやち敏', 'こやり隆史', 'ながえ孝子', 'ラサール石井', '三上えり', '三宅伸吾', '三浦信祐', '三浦靖', '上月良祐', '世耕弘成', '中条きよし', '串田誠一', '仁比聡平', '伊勢崎賢治', '伊波洋一', '倉林明子', '出川桃子', '初鹿野裕樹', '加田裕之', '勝部賢志', '北村晴男', '北村経夫', '古屋圭司', '古庄玄知', '司隆史', '嘉田由紀子', '堂故茂', '堂込麻紀子', '塩入清香', '塩村あやか', '塩田博昭', '大家敏志', '大椿ゆうこ', '大津力', '大門実紀史', '天畠大輔', '奥村政佳', '奥村祥大', '奥田ふみよ', '安江伸夫', '安達悠司', '安野貴博', '宮出千慧', '宮口治子', '宮本周司', '宮本和宏', '宮沢洋一', '小島とも子', '小沢雅仁', '小沼巧', '小西洋之', '岡崎太', '岩本剛人', '岩本麻奈', '岩渕友', '広田一', '庭田幸恵', '徳永エリ', '打越さく良', '新妻秀規', '新実彰平', '星北斗', '有村治子', '望月良男', '朝日健太郎', '木戸口英司'

In [27]:
members_tbl = build_member_table(df)          # 除外する前の df を使う
print(members_tbl["source"].value_counts())
members_tbl.to_csv("../data/external/member_names.csv", index=False)
print("保存しました:", len(members_tbl), "人")

source
guess     855
known     698
header    628
none        3
Name: count, dtype: int64
保存しました: 2184 人


In [ ]:
hits = q["text_nomask"].str.findall(SURNAME_RE)
pd.Series([h for lst in hits for h in lst]).value_counts().head(40)

In [2]:
import src.preprocess.filters as f
print("SURNAME_RE" in dir(f), "mask_person_names" in dir(f), "REPORT" in dir(f))
print("議員名の一覧:", "読み込み済み" if f.FULL_NAME_RE is not None else "なし")

True True True
議員名の一覧: 読み込み済み


In [3]:
# 会派 → 政党
GROUP2PARTY = {
    "自由民主党・無所属の会": "自民", "自由民主党": "自民",
    "立憲民主党・無所属": "立憲", "立憲民主・社民・無所属": "立憲", "立憲民主・無所属": "立憲",
    "中道改革連合・無所属": "中道", "公明党": "公明", "日本維新の会": "維新",
    "国民民主党・無所属クラブ": "国民", "国民民主党・新緑風会": "国民",
    "日本共産党": "共産", "れいわ新選組": "れいわ", "参政党": "参政",
    "日本保守党": "保守", "社会民主党": "社民",
    "チームみらい": "みらい", "チームみらい・無所属の会": "みらい",
}

# 読み込み
RAW = Path("../data/raw/kokkai")
dfs = []
for p in sorted(RAW.glob("speech_*.json")):
    with open(p, encoding="utf-8") as f:
        dfs.append(pd.DataFrame(json.load(f)["speechRecord"]))
df = pd.concat(dfs, ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M")

# 除外
q = filter_speeches(df)
q = q[q["date"] < "2026-08-01"].copy()

# 政党(本文を作る前に)
q["party"] = q["speakerGroup"].map(GROUP2PARTY).fillna("その他")

# 本文(マスクあり / なし)
q["text"] = [prepare_text(s, sp, p) for s, sp, p in zip(q["speech"], q["speaker"], q["party"])]
q["text_nomask"] = q["speech"].map(lambda s: prepare_text(s, mask=False))
print("全体:", len(df), "→ 除外後:", len(q))

全体: 194567 → 除外後: 88773


In [4]:
hits = q["text_nomask"].str.findall(SURNAME_RE)
pd.Series([h for lst in hits for h in lst]).value_counts().head(40)

高市     2327
石破     2031
今      1331
赤澤     1051
小泉      726
片山      516
加藤      482
茂木      475
松本      448
鈴木      427
林       372
上野      368
福岡      342
平       331
黄川田     328
城内      313
小野田     301
三原      292
岩屋      289
金子      280
武藤      275
江藤      270
伊東      270
坂井      265
岸田      253
中野      224
村上      219
中谷      168
あべ      166
木原      161
野田      146
古賀      127
あかま     123
浅尾      120
玉木      114
西田      110
高橋      108
牧野      107
石原      105
吉村      102
Name: count, dtype: int64

In [7]:
s = (q[q["party"] != "その他"]
     .sample(frac=1, random_state=0)
     .groupby("party")
     .head(3000))

vec = CountVectorizer(analyzer="char", ngram_range=(2, 4), min_df=20)
X = vec.fit_transform(s["text"])
terms = np.array(vec.get_feature_names_out())
content = np.array([bool(re.search(r"[一-龥ァ-ヶ]", t)) for t in terms])

for party in s["party"].value_counts().index:
    z = fightin_words(X, (s["party"] == party).to_numpy())
    z[~content] = -np.inf
    top = terms[np.argsort(z)[::-1][:100]]
    print(party, ":", "/".join(top), "\n")

維新 : 改革/無償化/償化/会保険料/社会保険/会保険/具合/いう具/う具/大阪/という具/無償/社会保/会保/改革を/具合に/革を/連立/険料/保険料/中国/我々/首都/高校/いう具合/う具合/う具合に/思うんで/役世/現役世/思うん/役世代/現役世代/尖閣/副首都/副首/と思うん/思う/我が党/が党/現役/合意書/意書/大学/給食/万博/と思う/医療費/療費/連立合意/立合/立合意/連立合/込ませ/の無償/役世代の/関して/に関して/関しては/の無償化/、大阪/険料を/保険料を/ち込ま/学術会議/学術会/術会議/術会/育機会/教育機会/保険/料を下/具合に思/合に思/料を下げ/、我々/険料を下/学術/関西/関税/処方/教育機/育機/割負担/割負/質問させ/革の/改革の/質問さ/関し/に関し/問させ/問させて/の社会保/お聞き/の無/お聞/意書に/、我/問さ 

立憲 : 修正/夫婦/大臣/年金/皆さ/皆さん/大臣、/臣、/定税/暫定税/暫定税率/定税率/の皆さ/の皆さん/、大臣、/、大臣/夫婦別/婦別/教職員/教科/。 十/ 十/彼女/政党、政/党、政党/党、政/な形態/委員/、政党、/の教科/の教科書/教科書/科書/努めるこ/形態の/修正案/決議/底上げ/底上/的夫婦別/的夫婦/的夫/暫定/る附/夫婦別姓/婦別姓/政党、/言っ/年金の/置を講ず/る附帯決/税率/る附帯/党、/教職/附帯/帯決/帯決議/附帯決議/附帯決/の暫定税/を講ずる/答え/択的夫/選択的夫/択的夫婦/ふうに思/この間/を講ず/、修正/生年金/厚生年/厚生年金/講ずる/講ず/措置を講/置を講/生年/択的/選択的/うに思/な措置を/会合/基金/に思いま/同組合/協同組合/に思/人身/定税率廃/別姓/年も/努める/電磁的/磁的/うに思い/成田/に思い/協同組/分か 

自民 : 我が国/地域/我が/が国/地元/歯科/国旗/我が国の/、地/が国の/整備/重要/地方/、我が国/感謝/申し上げ/申し上/し上/し上げ/取組/、我が/成長/産業/の地/申し/人口/し上げま/私の地/取り組/私の地元/、我/り組/いと存/いと存じ/たいと存/御指/地方創/地方創生/方創/方創生/農業/の地元/大変/上げま/国境離/境離/境離島/国境離島/要であ/健診/国の/水産/取り組ん/り組ん/り組んで/強靱/経済/思い

In [8]:
def show(party, pat, n=3):
    hit = s[(s["party"] == party) & s["text"].str.contains(pat)]
    print(f"[{party}] {pat}: {len(hit)}件")
    for t in hit["text"].head(n):
        i = t.find(pat)
        print("   …", t[max(0, i - 60): i + 30])

show("自民", "質疑は終了")
show("自民", "緊急集会")
show("立憲", "電磁的記録")
show("中道", "連合")          # ①の修正後は、ほぼ0件になるはず
show("れいわ", "政党")

for pat in ["政党、議員", "政党の議員", "我々政党", "政党は"]:
    print(pat, s[s["text"].str.contains(pat)]["party"].value_counts().head(5).to_dict())

[自民] 質疑は終了: 0件
[自民] 緊急集会: 18件
   …  緊急事態対応を考えるに当たりまして、参議院の緊急集会をめぐる主な論点を整理した上で、それぞれの論点につい
   … ておりますという問題が表層化していることに対しまして立法府としての意思を示す必要がありますし、さらには、緊急事態状況下で緊急集会が全国の声を集約する機能を発揮するにはいかなる憲法で
   … めていくことが基本であると考えております。 そこで、改めてワーキングチーム取りまとめについて説明をいたしますと、参議院の緊急集会の位置付けに関しましては、憲法において緊急事態に対応
[立憲] 電磁的記録: 20件
   … それから、何度もこれは指摘されているところですが、電磁的記録提供命令が取り消された場合でも、入手したデータやそ
   … 衆議院の法務委員会において、池田参考人が電磁的記録に着目した保管、管理の仕組みを設けられてよいと発言
   … 期間を明示しないで出される、差押えの対象になる電磁的記録もあるという答弁いただきました。 この電磁的記録が
[中道] 連合: 25件
   … からまた暫定予算なども視野に入れて、そして、国民の思いを入れた丁寧な審議が必要だと思っておりますけれども、この点について連合としてお考えをお願いします。
   … 得の公平な反映を目指す後期高齢者医療制度や国民健康保険の事務の実施について、デジタル技術の活用や都道府県国民健康保険団体連合会の活用を推進するなど自治体の事務負担の軽減に努めること
   … わったかというと、一般財団法人ですよ、孝情教育文化財団、そこに、目的の中に、前回まで目的には、当法人は、世界平和統一家庭連合が主唱する理念に基づき、つまり、統一教会の理念に基づきこ
[れいわ] 政党: 497件
   …  裏金問題が発覚してからもう一年半が経過しています。なのに、何も明らかになっていないんですよ。政党はこの参考人招致にも反対し、そして、全くこの裏金問題を反
   … いう言葉も伺いましたけれども、その濃厚さを今日は一緒に考えていきたい、確認をしていきたいなと思っております。 私たちは、政党、積極財政を訴える政党ですから、いわゆる積極財政とは足り
   … たんですが、今日は保険業法の質疑、もう時間ないですけれども、最後に一問聞きます。 

In [4]:
m = q["speech"].str.contains("質疑は終了いたしました")
head = q.loc[m, "speech"].str.extract(r"^(○\S+?)[\s　]")[0]
print(m.sum(), "件")
print(head.str.replace(r"（[^）]*）", "", regex=True).str[-4:].value_counts().head(15))

183 件
0
主査代理    25
田所主査    21
深澤主査    21
牧島主査    20
山下主査    19
小林主査    19
高木主査    18
齋藤主査    17
河西主査    14
○理事      5
安住座長     1
井上座長     1
坂本座長     1
齋藤座長     1
Name: count, dtype: int64


In [6]:
# ① 司会役(主査・理事・座長)が消えたか
m = q["speech"].str.contains("質疑は終了いたしました")
print("質疑は終了:", m.sum(), "件")

# ② 自己紹介と「我々政党」が消えたか
for pat in ["政党の議員です", "政党、議員です", "政党の議員でございます", "我々政党"]:
    print(pat, q["text"].str.contains(pat).sum(), "件")

# ③ 自己紹介を消しすぎていないか(例を3件)
hit = q[q["text_nomask"].str.contains("の議員の皆さん")]
for t in hit["text"].head(3):
    i = t.find("議員の皆さん")
    print("   …", t[max(0, i - 40): i + 20])

質疑は終了: 0 件
政党の議員です 0 件
政党、議員です 0 件
政党の議員でございます 0 件
我々政党 0 件
   … は、積極的に不記載の指示はしていないと言っている。一方で、政倫審に出られた多くの議員の皆さんは、指示を受けてやった、秘書
   … すけれども。 大臣、十年後も恐らくこの審議をきっとするんだと思います。そのときの議員の皆さんが、この十年間の新たな受給権
   … と出してこの成果を国民の皆さんに示すことが大事なんだというようなこともたくさんの議員の皆さんから言われました。 その中で


In [4]:
print("主査" in f.CHAIR)
print(prepare_text("○田中委員　自由民主党の田中太郎です。我々日本維新の会は反対です。", "田中太郎", "自民"))

True
我々は反対です。


In [4]:
import pandas as pd

GROUP2PARTY = {
    "自由民主党・無所属の会": "自民", "自由民主党": "自民",
    "立憲民主党・無所属": "立憲", "立憲民主・社民・無所属": "立憲", "立憲民主・無所属": "立憲",
    "中道改革連合・無所属": "中道", "公明党": "公明", "日本維新の会": "維新",
    "国民民主党・無所属クラブ": "国民", "国民民主党・新緑風会": "国民",
    "日本共産党": "共産", "れいわ新選組": "れいわ", "参政党": "参政",
    "日本保守党": "保守", "社会民主党": "社民",
    "チームみらい": "みらい", "チームみらい・無所属の会": "みらい",
}
pd.DataFrame(list(GROUP2PARTY.items()), columns=["speakerGroup", "party"]) \
  .to_csv("../data/external/group_party_map.csv", index=False, encoding="utf-8")
print(pd.read_csv("../data/external/group_party_map.csv"))

    speakerGroup party
0    自由民主党・無所属の会    自民
1          自由民主党    自民
2      立憲民主党・無所属    立憲
3    立憲民主・社民・無所属    立憲
4       立憲民主・無所属    立憲
5     中道改革連合・無所属    中道
6            公明党    公明
7         日本維新の会    維新
8   国民民主党・無所属クラブ    国民
9     国民民主党・新緑風会    国民
10         日本共産党    共産
11        れいわ新選組   れいわ
12           参政党    参政
13         日本保守党    保守
14         社会民主党    社民
15        チームみらい   みらい
16  チームみらい・無所属の会   みらい


In [5]:
q = pd.read_parquet("../data/processed/speeches.parquet")
q["month"] = q["date"].dt.to_period("M")